# Deploy the hosted Copilot SDK agent

This notebook stands up the `github-copilot` agent end to end using **Azure CLI + the `azure-ai-projects` Python SDK**, the same pattern as every other example in this repo (see [`08-03-hosted-agents`](../08-03-hosted-agents/08-03-01-deploy-hosted-agent.ipynb)).

Every action is explicit and visible - the notebook drives the deployment directly with the Azure CLI and SDK.

By the end you will have:

- A new AI Foundry account + project provisioned in `swedencentral` from `infra/main.bicep` (via `az deployment sub create`).
- A `gpt-5.4-mini` GlobalStandard model deployment in the project.
- The Copilot SDK agent container built and pushed via `az acr build`, then registered as a hosted agent (`AgentProtocol.INVOCATIONS`) via `AIProjectClient.agents.create_version`.
- The per-agent managed identity granted `AcrPull` on the ACR plus the data-plane roles it needs to pull the image and call the model.
- Two smoke-test invocations against the deployed agent over `POST .../endpoint/protocols/invocations`.
- A CSV analytics demo: upload `data/m365-licenses.csv` plus a `data/m365-reference.json` (SKU costs + department names) to a session sandbox, then run five M365 license analytics prompts through the agent using the [session file-ops REST API](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions?pivots=rest#session-file-operations).
- OpenTelemetry traces visible in **Foundry portal -> Tracing**.

This folder bundles the subscription-scoped Bicep (`infra/`), the agent container (`src/github-copilot-invocations/`), and a synthetic dataset (`data/m365-licenses.csv`). See [`08-10-00-hosted-copilot-sdk-agent.md`](08-10-00-hosted-copilot-sdk-agent.md) for what each file in this folder does.

## Prerequisites

1. **`az` CLI** logged in: `az login` (the prereq cell asserts this).
2. **Python packages** (provided by the repo's `uv` environment): `azure-ai-projects>=2.1.0`, `azure-identity`, `requests`.
3. **Quota** for `gpt-5.4-mini` GlobalStandard in `swedencentral` (or change `LOCATION` below and the bicep `@allowed` list).

Run this notebook from the folder it lives in (`08-agents/08-10-hosted-copilot-sdk-agent/`) so the relative paths to `infra/`, `src/`, and `data/` resolve.

In [ ]:
import json
import pathlib
import shutil
import subprocess

def sh(cmd, **kw):
    """Run a shell command, stream output, return CompletedProcess."""
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=False, **kw)

def _require(binary, install_hint):
    if shutil.which(binary) is None:
        raise AssertionError(
            f"`{binary}` is not on PATH. Install it and restart the kernel:\n  {install_hint}"
        )

_require("az", "Ubuntu: https://learn.microsoft.com/cli/azure/install-azure-cli-linux")

sh("az --version | head -1")

# Require az login
who = subprocess.run(
    "az account show --query '{name:name, id:id, tenant:tenantId, user:user.name}' -o jsonc",
    shell=True, capture_output=True, text=True,
)
if who.returncode != 0:
    raise AssertionError(
        "Not logged in to Azure CLI. Run `az login` in a terminal, then re-run this cell.\n"
        f"az stderr: {who.stderr.strip()}"
    )
print(who.stdout)


## Step 1 - Variables and principal ID

Pick a short, lowercase **environment name** - it stamps the `azd-env-name` tag on every resource and names the resource group (`rg-<env>`). The Foundry **account** and **project** follow this repo's convention - `aif-copilot-sdk-<suffix>` and `project-copilot-sdk-<suffix>` - where the 6-char suffix (a hash of the subscription + environment name) keeps the globally-unique account FQDN distinct across developers.

The current caller's principal ID (extracted from the JWT, not via a Graph call) is passed to bicep as `principalId` so role assignments target the right identity.

In [ ]:
import base64
import hashlib

ENV_NAME = "foundry-copilot-sdk-08-10"
LOCATION = "swedencentral"
DEPLOYMENT_NAME = "gpt-5.4-mini"
AGENT_NAME = "github-copilot"
RESOURCE_GROUP = f"rg-{ENV_NAME}"

# Subscription ID - used to build resource IDs for the role grants in Step 5
SUBSCRIPTION_ID = subprocess.run(
    "az account show --query id -o tsv",
    shell=True, capture_output=True, text=True,
).stdout.strip()

# 6-char suffix for globally-unique resource names, following this repo's
# convention (e.g. aif-spoke-alpha-c2676f) - the account FQDN must be globally unique.
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + ENV_NAME).encode()).hexdigest()[:6]
AI_ACCOUNT_NAME = f"aif-copilot-sdk-{SUFFIX}"
AI_PROJECT_NAME = f"project-copilot-sdk-{SUFFIX}"

# Principal ID from JWT - avoids a Graph round-trip
token = subprocess.run(
    "az account get-access-token --query accessToken -o tsv",
    shell=True, capture_output=True, text=True,
).stdout.strip()
PRINCIPAL_ID = json.loads(base64.urlsafe_b64decode(token.split('.')[1] + '=='))['oid']
PRINCIPAL_TYPE = "User"

print(f"Subscription:   {SUBSCRIPTION_ID}")
print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location:       {LOCATION}")
print(f"Env name:       {ENV_NAME}")
print(f"AI account:     {AI_ACCOUNT_NAME}")
print(f"AI project:     {AI_PROJECT_NAME}")
print(f"Agent name:     {AGENT_NAME}")
print(f"Model:          {DEPLOYMENT_NAME}")
print(f"Principal ID:   {PRINCIPAL_ID}")


## Step 2 - Provision the Foundry stack with Bicep

`infra/main.bicep` is subscription-scoped, so we use `az deployment sub create`. The bicep:

1. Creates resource group `rg-<env-name>`.
2. Provisions an AI Services account + a Foundry project.
3. Deploys the model named in `DEPLOYMENT_NAME` (GlobalStandard, capacity 100).
4. Creates an Azure Container Registry connected to the project.
5. Creates Application Insights + Log Analytics for tracing.
6. Provisions the **capability host** (the hosted-agent runtime) because `enableCapabilityHost=true` is passed.

We assemble the deployment parameters in code from the values above and write them to a file next to the template, then pass it to `az deployment sub create`.

Provisioning typically takes **5-10 minutes**. Watch the stream below; on success, the cell after this one extracts the bicep outputs we need for the next steps.


In [ ]:
# Build the bicep model-deployment parameter we pass to the template.
ai_project_deployments = [
    {
        "name": DEPLOYMENT_NAME,
        "model": {
            "format": "OpenAI",
            "name": DEPLOYMENT_NAME,
            "version": "2026-03-17",
        },
        "sku": {
            "name": "GlobalStandard",
            "capacity": 100,
        },
    }
]

bicep_params = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "environmentName":                 {"value": ENV_NAME},
        "location":                        {"value": LOCATION},
        "aiDeploymentsLocation":           {"value": LOCATION},
        "aiFoundryResourceName":           {"value": AI_ACCOUNT_NAME},
        "aiFoundryProjectName":            {"value": AI_PROJECT_NAME},
        "principalId":                     {"value": PRINCIPAL_ID},
        "principalType":                   {"value": PRINCIPAL_TYPE},
        "aiProjectDeploymentsJson":        {"value": json.dumps(ai_project_deployments)},
        "enableHostedAgents":              {"value": True},
        "enableCapabilityHost":            {"value": True},
        "enableMonitoring":                {"value": True},
    },
}

PARAMS_FILE = pathlib.Path("infra") / ".main.parameters.runtime.json"
PARAMS_FILE.write_text(json.dumps(bicep_params, indent=2))
print(f"Wrote {PARAMS_FILE}")


In [ ]:
# Subscription-scope deployment - this is the slow one (5-10 min). Streams output live.
DEPLOY_NAME = f"deploy-{ENV_NAME}"
sh(
    f'az deployment sub create '
    f'--name "{DEPLOY_NAME}" '
    f'--location "{LOCATION}" '
    f'--template-file infra/main.bicep '
    f'--parameters @"{PARAMS_FILE}" '
    f'-o table'
)


### Extract bicep outputs

We need the project endpoint, ACR endpoint, App Insights connection string, and the account/project names for the rest of the notebook.


In [ ]:
r = subprocess.run(
    f'az deployment sub show --name "{DEPLOY_NAME}" --query properties.outputs -o json',
    shell=True, capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
raw_outputs = json.loads(r.stdout)
outputs = {k.upper(): v["value"] for k, v in raw_outputs.items()}

PROJECT_ENDPOINT       = outputs["AZURE_AI_PROJECT_ENDPOINT"]
ACR_LOGIN_SERVER       = outputs["AZURE_CONTAINER_REGISTRY_ENDPOINT"]
ACR_NAME               = ACR_LOGIN_SERVER.split(".")[0]
ACCOUNT_NAME           = outputs["AZURE_AI_ACCOUNT_NAME"]
PROJECT_NAME           = outputs["AZURE_AI_PROJECT_NAME"]
APPINSIGHTS_CONN_STR   = outputs.get("APPLICATIONINSIGHTS_CONNECTION_STRING", "")

print(f"Project endpoint: {PROJECT_ENDPOINT}")
print(f"Account name:     {ACCOUNT_NAME}")
print(f"Project name:     {PROJECT_NAME}")
print(f"ACR login server: {ACR_LOGIN_SERVER}")
print(f"ACR name:         {ACR_NAME}")
print(f"App Insights:     {APPINSIGHTS_CONN_STR[:60]}{'...' if len(APPINSIGHTS_CONN_STR) > 60 else ''}")


## Step 3 - Build the agent image with `az acr build`

ACR remote build - same pattern as 08-03 (`az acr build --registry $ACR --platform linux/amd64 ./agent-dir/`). The platform flag matters: Foundry's hosted runtime is `linux/amd64`, so Apple Silicon hosts must not produce native ARM images.

A first build takes about **2-4 minutes**.


In [ ]:
print(f"Building {AGENT_NAME}:latest for linux/amd64 on ACR {ACR_NAME}...")
sh(
    f'az acr build --registry "{ACR_NAME}" '
    f'--image "{AGENT_NAME}:latest" '
    f'--platform linux/amd64 '
    f'./src/github-copilot-invocations/'
)


## Step 4 - Register the agent as a hosted version

Use `AIProjectClient.agents.create_version` (same SDK call as 08-03 - the only difference is the protocol). 08-03 registers with `AgentProtocol.RESPONSES` because its container speaks Microsoft Agent Framework\'s `ResponsesHostServer`; this agent registers with `AgentProtocol.INVOCATIONS` because `main.py` uses `azure.ai.agentserver.invocations.InvocationAgentServerHost`.

**Two-pass registration.** Each hosted-agent version metadata exposes two identities: `AgentIdentityBlueprint` (a *template* used by the platform to provision per-version identities) and `AgentIdentity` (`instance_identity` in the API) which is the **actual Entra service principal the container assumes at runtime**. The blueprint exists at registration time, but the container ALWAYS uses the AgentIdentity for outbound calls. So:

- We pin `AZURE_CLIENT_ID` to `instance_identity.client_id` (the AgentIdentity, not the blueprint) so `DefaultAzureCredential` inside the container resolves to the runtime SP.
- Step 5 then grants the AgentIdentity\'s principal the RBAC roles it needs.

We register a bootstrap v1 to learn the AgentIdentity's client_id, register v2 with `AZURE_CLIENT_ID` set, then delete the bootstrap so the agent is left with a single version.

We inject the env vars the container needs at startup:

- `AZURE_AI_PROJECT_ENDPOINT` - the Foundry project endpoint. `main.py` appends `/openai/v1/` and points the Copilot SDK's `base_url` at `<project-endpoint>/openai/v1/`, reaching the model over the project's own OpenAI-compatible surface (token audience `ai.azure.com`). The platform also auto-injects this as `FOUNDRY_PROJECT_ENDPOINT`.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` - the model deployment name we want as BYOK.
- `AZURE_CLIENT_ID` - the AgentIdentity client_id (set in Step 4.5).


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential
from azure.core.rest import HttpRequest

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

CONTAINER_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:latest"

# Bootstrap registration. The container will not be invokable yet because
# AZURE_CLIENT_ID is unset and roles will not be granted until Step 5, but
# this gives us the per-version AgentIdentity client_id that we need in 4.5.
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.INVOCATIONS, version="1.0.0"),
        ],
        # 2 vCPU / 4Gi: the Copilot SDK runs multi-step shell+python turns (Step 7) that recycle a 1 CPU / 2Gi container.
        cpu="2",
        memory="4Gi",
        image=CONTAINER_IMAGE,
        environment_variables={
            "AZURE_AI_PROJECT_ENDPOINT":       PROJECT_ENDPOINT,
            "AZURE_AI_MODEL_DEPLOYMENT_NAME":  DEPLOYMENT_NAME,
        },
    ),
)
print(f"Bootstrap version registered: {agent.name} v{agent.version}")


### Step 4.5 - Re-register with `AZURE_CLIENT_ID` pinned

A Foundry hosted agent runs the container with two managed identities attached (`blueprint` + per-version `instance_identity`). Inside the container, `DefaultAzureCredential` has no way to pick between them and throws on `.get_token()`, which surfaces as a generic 500 with body `Internal Server Error`.

The workaround is to pin `AZURE_CLIENT_ID` to the `instance_identity.client_id`. But we only know that ID after registering, so we read it back from the bootstrap version and register a new version with it set. We then delete the bootstrap version (`delete_version`) so the agent is left with just the invokable v2.


In [ ]:
# Read the runtime AgentIdentity\'s client_id from the bootstrap version.
# Note: the metadata calls this `instance_identity`. The `blueprint` field is
# a TEMPLATE identity used by the platform for provisioning, not the runtime
# SP. Roles must be granted to AgentIdentity (this field), not Blueprint.
bootstrap_version = agent.version  # v1 from Step 4; capture before agent is reassigned to v2 below
v_meta = json.loads(
    client.send_request(
        HttpRequest("GET", f"/agents/{AGENT_NAME}/versions/{agent.version}?api-version=v1")
    ).text()
)
AGENT_IDENTITY_CLIENT_ID = v_meta["instance_identity"]["client_id"]
print(f"Pinning AZURE_CLIENT_ID = {AGENT_IDENTITY_CLIENT_ID} (AgentIdentity)")

# Register the version we will actually invoke
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.INVOCATIONS, version="1.0.0"),
        ],
        cpu="2",
        memory="4Gi",
        image=CONTAINER_IMAGE,
        environment_variables={
            "AZURE_AI_PROJECT_ENDPOINT":      PROJECT_ENDPOINT,
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": DEPLOYMENT_NAME,
            "AZURE_CLIENT_ID":                AGENT_IDENTITY_CLIENT_ID,
        },
    ),
)
print(f"Active version: {agent.name} v{agent.version}")
print(f"Image:          {CONTAINER_IMAGE}")

# The bootstrap (v1) existed only to surface the AgentIdentity client_id pinned above.
# v2 now carries AZURE_CLIENT_ID and is the version we invoke, so delete v1 to leave the
# agent with a single clean version. (The per-agent identity is stable across versions, so
# dropping v1 does not affect v2.)
client.agents.delete_version(AGENT_NAME, agent_version=bootstrap_version)
print(f"Deleted bootstrap v{bootstrap_version}; agent now has the single version v{agent.version}")


## Step 5 - Grant runtime roles to the per-agent managed identity

Foundry creates a **per-agent managed identity** (the AgentIdentity from Step 4) at registration time, and the container assumes it at runtime. We grant it four roles:

- `AcrPull` on the **ACR** - so the platform can pull the image.
- `Foundry User` (`53ca6127-...`, formerly "Azure AI User") on the **project** - the role that enables the model call: `main.py` reaches `<project-endpoint>/openai/v1/responses` with the `ai.azure.com` audience, which maps to project data-plane access.
- `Cognitive Services OpenAI User` (`5e0bd9bd-...`) on the **account** - a safety net left over from the earlier account-endpoint design.
- `Cognitive Services User` (`a97b65f3-...`) on the **account** - a broader safety net; once the agent works against the project endpoint you can drop these two account-scoped roles.

We read the AgentIdentity's principal ID from the version metadata via the SDK's raw-request escape hatch, then use `az role assignment create`.

In [ ]:
from azure.core.rest import HttpRequest

# Re-read the AgentIdentity principal from the active version (same as 4.5).
v = json.loads(
    client.send_request(
        HttpRequest("GET", f"/agents/{AGENT_NAME}/versions/{agent.version}?api-version=v1")
    ).text()
)
AGENT_IDENTITY_PRINCIPAL_ID = v["instance_identity"]["principal_id"]
print(f"AgentIdentity principal: {AGENT_IDENTITY_PRINCIPAL_ID}")

acr_id = subprocess.check_output(
    f"az acr show -n {ACR_NAME} --query id -o tsv", shell=True, text=True,
).strip()
account_id = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
)
project_id = f"{account_id}/projects/{PROJECT_NAME}"

# Role IDs:
FOUNDRY_USER                 = "53ca6127-db72-4b80-b1b0-d745d6d5456d"   # general Foundry data plane
COG_SVCS_OPENAI_USER         = "5e0bd9bd-7b93-4f28-af87-19fc36ad61bd"   # OpenAI deployments/responses/*
COG_SVCS_USER                = "a97b65f3-24c7-4388-baec-2e87135dc908"   # generic Cognitive Services data

grants = [
    ("AcrPull",              acr_id),       # pull the container image
    (FOUNDRY_USER,           project_id),   # general project data plane
    (COG_SVCS_OPENAI_USER,   account_id),   # /openai/v1/responses on the account
    (COG_SVCS_USER,          account_id),   # broader Cognitive Services data plane (safety net)
]
for role, scope in grants:
    r = subprocess.run(
        f"az role assignment create --assignee-object-id {AGENT_IDENTITY_PRINCIPAL_ID} "
        f"--assignee-principal-type ServicePrincipal --role '{role}' --scope '{scope}'",
        shell=True, capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f"  granted: {role}")
    elif "already exist" in r.stderr.lower():
        print(f"  already granted: {role}")
    else:
        print(f"  failed:  {role} - {r.stderr.strip()[:200]}")

print()
print("Waiting 90s for RBAC + container cold start to settle...")
import time; time.sleep(90)
print("Done. Continue to Step 6.")


## Step 6 - Smoke-test the agent

The agent speaks the Invocations protocol over `POST .../endpoint/protocols/invocations`. We build a small `invoke()` helper that POSTs `{"input": "..."}`, parses the SSE stream, accumulates the assistant deltas, and captures the `session_id` from the terminal event that carries it. The same helper is reused in Step 7 for the CSV demo - session threading there is just `invoke(..., session_id=SESSION_ID)`.


In [ ]:
import sys
import time
import requests
from IPython.display import Markdown, display

API_VERSION = "v1"

def _bearer():
    return DefaultAzureCredential().get_token("https://ai.azure.com/.default").token

def invoke(input_text, session_id=None, stream=True, render=False, retries=5, retry_delay=30):
    """POST to the agent. Streams SSE assistant deltas; returns (text, session_id).

    With render=True the reply is rendered live as Markdown (so tables become real
    tables) instead of raw streamed text - use it for the analytics prompts below.

    Retries on 5xx and on mid-stream `error` events (the container warming up,
    RBAC propagating, or being recycled mid-turn). Default budget is 5 retries x 30s = 150s, enough to cover
    typical RBAC propagation + container cold start. On final failure, prints
    the response body before raising.
    """
    base = (
        f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/protocols/"
        f"invocations?api-version={API_VERSION}"
    )
    url = base if session_id is None else f"{base}&agent_session_id={session_id}"
    body = json.dumps({"input": input_text})
    last_status, last_body = None, None
    md_handle = display(Markdown(""), display_id=True) if render else None
    for attempt in range(1, retries + 1):
        headers = {
            "Authorization": f"Bearer {_bearer()}",
            "Foundry-Features": "HostedAgents=V1Preview",
            "Content-Type": "application/json",
        }
        resp = requests.post(url, headers=headers, data=body, stream=True)
        if resp.status_code >= 500 and attempt < retries:
            # Read body now (before stream is consumed) so we can show it if needed.
            last_status, last_body = resp.status_code, resp.text[:500]
            print(
                f"[invoke] {resp.status_code} from server (attempt {attempt}/{retries}); "
                f"retrying in {retry_delay}s..."
            )
            time.sleep(retry_delay)
            continue
        if not resp.ok:
            print(f"[invoke] HTTP {resp.status_code}")
            print(f"[invoke] body: {resp.text[:1500]}")
            resp.raise_for_status()
        # Success - stream SSE
        chunks, out_sid, stream_error = [], session_id, None
        for raw in resp.iter_lines(decode_unicode=True):
            if not raw or not raw.startswith("data:"):
                continue
            try:
                event = json.loads(raw[5:].strip())
            except json.JSONDecodeError:
                continue
            etype = event.get("type")
            if etype == "assistant.message_delta":
                delta = event.get("data", {}).get("deltaContent") or ""
                chunks.append(delta)
                if render:
                    if "\n" in delta:
                        md_handle.update(Markdown("".join(chunks)))
                elif stream:
                    sys.stdout.write(delta)
                    sys.stdout.flush()
            elif etype == "error":
                stream_error = event.get("message")
                break
            elif "session_id" in event and "invocation_id" in event:
                out_sid = event["session_id"]
        # A mid-stream error (container recycled, gateway reset, etc.) means the turn
        # produced no usable answer: retry if attempts remain, else fail loudly.
        if stream_error:
            if attempt < retries:
                last_status, last_body = "stream-error", stream_error[:500]
                print(f"\n[invoke] agent error mid-stream (attempt {attempt}/{retries}); retrying in {retry_delay}s...")
                time.sleep(retry_delay)
                continue
            print(f"\n[agent error] {stream_error}", file=sys.stderr)
            raise RuntimeError(f"Agent invocation failed after {retries} attempts (mid-stream error): {stream_error[:200]}")
        if render:
            md_handle.update(Markdown("".join(chunks)))
        elif stream:
            print()
        return "".join(chunks), out_sid
    # Exhausted retries
    print(f"[invoke] exhausted {retries} retries. Last status: {last_status}")
    print(f"[invoke] last body: {last_body}")
    raise RuntimeError(f"Agent invocation failed after {retries} retries (status={last_status}).")

def upload_session_file(session_id, local_path, target_path):
    """PUT a local file into the session sandbox."""
    url = (
        f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/sessions/{session_id}"
        f"/files/content?api-version={API_VERSION}&path={target_path}"
    )
    headers = {
        "Authorization": f"Bearer {_bearer()}",
        "Foundry-Features": "HostedAgents=V1Preview",
        "Content-Type": "application/octet-stream",
    }
    with open(local_path, "rb") as f:
        r = requests.put(url, headers=headers, data=f.read())
    r.raise_for_status()
    return r


def download_session_file(session_id, remote_path, local_path):
    """GET a file out of the session sandbox (the read counterpart to upload_session_file)."""
    url = (
        f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/sessions/{session_id}"
        f"/files/content?api-version={API_VERSION}&path={remote_path}"
    )
    headers = {
        "Authorization": f"Bearer {_bearer()}",
        "Foundry-Features": "HostedAgents=V1Preview",
    }
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    with open(local_path, "wb") as f:
        f.write(r.content)
    return len(r.content)


def show_agent_status():
    """Pretty-print the current agent version status from the project data plane."""
    from azure.core.rest import HttpRequest
    raw = client.send_request(
        HttpRequest("GET", f"/agents/{AGENT_NAME}/versions/{agent.version}?api-version={API_VERSION}")
    ).text()
    info = json.loads(raw)
    # Surface the bits that actually matter for "is it healthy"
    summary = {
        "name":              info.get("name"),
        "version":           info.get("version"),
        "status":            info.get("status"),
        "provisioning_state": info.get("provisioning_state"),
        "image":             info.get("definition", {}).get("image"),
        "instance_identity": info.get("instance_identity"),
        "endpoint":          info.get("endpoint"),
        "last_error":        info.get("last_error") or info.get("error"),
    }
    print(json.dumps(summary, indent=2, default=str))
    return info

print("Helpers defined: invoke(), upload_session_file(), download_session_file(), show_agent_status().")


In [ ]:
# Smoke test 1 - general capability question.
invoke("What can you help me with? Be brief.")


In [ ]:
# Smoke test 2 - exercises the agent's built-in shell/Python tools (no CSV needed).
invoke("Use your shell tools to print the current date and your Python version, then summarise in one line.")

### Troubleshooting cold starts

The first invocations can return `500 Internal Server Error` while the container cold-starts and RBAC propagates - `invoke()` already retries through that window. If it keeps failing, inspect the deployed version directly: `provisioning_state`, `last_error`, and the resolved `instance_identity` are the fields that say whether it is still starting, mis-identified, or missing a role.

In [ ]:
# Run this if a smoke test keeps returning 500 after invoke() exhausts its retries.
show_agent_status()

## Step 7 - CSV analytics: M365 license cleanup demo

Hosted-agent sessions give every conversation a **persistent sandbox** with `$HOME` and an uploaded-files area. The Copilot SDK has shell + Python tools built-in, so once we upload a file, the agent can `awk` / `python` / `jq` over it across as many turns as we want.

### How the CSV reaches the agent (two distinct things)

**1. The CSV is uploaded to the session sandbox** in 7.2 via a single `PUT` call:

```
PUT  {PROJECT_ENDPOINT}/agents/github-copilot/endpoint/sessions/{SESSION_ID}/files/content
     ?api-version=v1
     &path=m365-licenses.csv
Content-Type: application/octet-stream
<raw file bytes>
```

The file now lives in the agent container\'s session-scoped filesystem (under `$HOME` or `/files/`, depending on platform mapping).

**2. The agent is _told_ to use it** by passing `session_id=SESSION_ID` on every subsequent `invoke()` call, which adds `?agent_session_id={SESSION_ID}` to the invocation URL. That query parameter is what routes the request to the same sandbox that has the CSV in it. The prompt body just says "Using the CSV `m365-licenses.csv`, ..." - the agent uses its built-in shell tool to `cat` / `awk` / `pandas` the file on its own filesystem.

**The prompt never contains the CSV.** The CSV is uploaded once, sits in the sandbox, and every prompt is just a sentence that references its filename. Drop the `session_id=SESSION_ID` argument (or pass a different one) and the agent lands in a fresh sandbox that does not have the file - it can\'t answer.

### What the cells below do

1. Make a warm-up invocation to create a session. The terminal `done` SSE event carries `session_id`.
2. Upload `data/m365-licenses.csv` (100 synthetic users, real Microsoft license SKUs, engineered outliers) **and** `data/m365-reference.json` (per-SKU costs + department-code names) into that session via the [REST file-ops endpoint](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions?pivots=rest#session-file-operations). The skill supplies the analysis *method*; this reference supplies the *data*.
3. Bind every follow-up invocation to that session ID with `?agent_session_id=<id>` so the sandbox keeps the file across turns.
4. Ask the agent five analytics questions - simple counts, filtered lookups, cross-filtered cost analysis, and an optimisation recommendation - and stream each answer inline.
5. Have the agent render a cost-by-department chart, save it into the sandbox, and download it back with the `download_session_file()` GET helper - the read counterpart to the upload in step 2.

The dataset\'s columns: `UserPrincipalName, sAMAccountName, DisplayName, Ext.5 (license SKU), Ext.6 (department code 1-8), Ext.3 (cost-centre flag), whenCreated, LastSignInDateTime, AccountEnabled`. Engineered outliers include 5 disabled-but-licensed accounts, 6 SPE_E5 holders inactive 90+ days, and several never-signed-in ghosts.


### 7.1 Warm-up - create a session, capture its `session_id`

In [ ]:
print("--- warm-up ---")
_, SESSION_ID = invoke(
    "We're about to analyse an M365 license CSV. Reply with one word: ready."
)
print(f"\nsession_id = {SESSION_ID}\n")


### 7.2 Upload the CSV and the cost/department reference into the session sandbox

In [ ]:
CSV_LOCAL = "data/m365-licenses.csv"
CSV_REMOTE = "m365-licenses.csv"
REF_LOCAL = "data/m365-reference.json"
REF_REMOTE = "m365-reference.json"

for local, remote in [(CSV_LOCAL, CSV_REMOTE), (REF_LOCAL, REF_REMOTE)]:
    upload_session_file(SESSION_ID, local, remote)
    print(f"Uploaded {local} ({pathlib.Path(local).stat().st_size} bytes) -> session {SESSION_ID} as {remote}")
print()

print("--- agent confirms it can see the files ---")
_ = invoke(
    f"Two files were just uploaded to your session: the data `{CSV_REMOTE}` and the reference `{REF_REMOTE}`. "
    "Locate them (try `find $HOME /files -name '*.csv' -o -name '*.json' 2>/dev/null`), run `head -2` on the CSV, "
    "and print the top-level keys of the JSON. Reply with the two paths and the CSV header line. Be terse.",
    session_id=SESSION_ID,
)

### 7.3 Prompt 1 - License count per SKU

In [ ]:
print("\n=== Q1: License count per SKU ===")
_ = invoke(
    f"Using the CSV `{CSV_REMOTE}`, return a markdown table of users per license SKU (column `Ext.5`), "
    "sorted by count descending. Columns: License SKU, User count. No commentary, table only.",
    session_id=SESSION_ID,
    render=True,
)


### 7.4 Prompt 2 - Stale SPE_E5 holders (90+ days inactive)

In [ ]:
print("\n=== Q2: SPE_E5 holders inactive 90+ days ===")
_ = invoke(
    f"Using `{CSV_REMOTE}`, list every user whose `Ext.5` is `SPE_E5` AND `AccountEnabled` is `TRUE` "
    "AND `LastSignInDateTime` is non-empty AND older than 90 days from today (2026-05-28). "
    "Return a markdown table: UserPrincipalName, DisplayName, LastSignInDateTime, days_since_signin, Ext.6. "
    "Sort by days_since_signin descending. End with a one-line summary of total count.",
    session_id=SESSION_ID,
    render=True,
)


### 7.5 Prompt 3 - Disabled accounts still holding a license

In [ ]:
print("\n=== Q3: Disabled accounts still licensed ===")
_ = invoke(
    f"Using `{CSV_REMOTE}`, list every row where `AccountEnabled` is `FALSE`. "
    "Return a markdown table: UserPrincipalName, Ext.5 (license SKU), Ext.6 (department code), whenCreated. "
    "End with: 'Cleanup candidates: <count>'.",
    session_id=SESSION_ID,
    render=True,
)


### 7.6 Prompt 4 - Cross-filter: monthly cost per department

The costs and department-code map are no longer in the prompt *or* the skill - they come from the uploaded `m365-reference.json`. The `m365-license-analytics` skill supplies only the analysis method.

In [ ]:
print("\n=== Q4: Monthly M365 cost per department ===")
_ = invoke(
    f"Using `{CSV_REMOTE}` joined to the costs and department names in `{REF_REMOTE}`, "
    "compute total monthly license spend per department. Return a markdown table sorted by "
    "total cost descending: Department, User count, Monthly cost (USD). End with the grand total.",
    session_id=SESSION_ID,
    render=True,
)


### 7.7 Prompt 5 - Optimisation recommendation (reclaim plan)

Likewise, the per-SKU costs come from `m365-reference.json`; the reclaim-rule definition comes from the skill.

In [ ]:
print("\n=== Q5: Reclaim plan - who to deprovision and how much we'd save ===")
_ = invoke(
    f"Using `{CSV_REMOTE}`, find every reclaim candidate per your reclaim rules (today is 2026-05-28), "
    f"costing each license from `{REF_REMOTE}`. "
    "Output exactly:\n"
    "1. A one-line summary: 'Total reclaimable: <N> users, <amount> USD/month'.\n"
    "2. A markdown table sorted by Monthly savings descending: "
    "UserPrincipalName, Reason, License (Ext.5), Monthly savings (USD).\n"
    "Be precise about the math.",
    session_id=SESSION_ID,
    render=True,
)


### 7.8 - Visualise: chart cost by department, then download it from the agent

The session sandbox is read **and** write, so the agent can produce artefacts we pull back out. Here it computes monthly cost per department (as in 7.6), renders it to a chart **file** saved inside the sandbox, and we retrieve that file with a `download_session_file()` GET helper - the read counterpart to the `upload_session_file()` PUT from 7.2.

The agent picks its own tooling. We prompt it to use **matplotlib**, which it will `pip install` into its own session shell if the package is not already present; if the container has no package egress, it falls back to hand-writing a dependency-free **SVG** bar chart. Either way it saves the chart under a known filename and reports it on a `CHART_FILE:` line that we parse to know what to download. If the file API cannot serve an agent-written file on your platform build, the cell falls back to having the agent return the SVG inline.

In [ ]:
import re
from IPython.display import Image, SVG, display

print("\n=== Q6: render a cost-by-department chart and save it to the sandbox ===")
chart_answer, _ = invoke(
    f"Using `{CSV_REMOTE}` joined to `{REF_REMOTE}`, compute total monthly license spend per department "
    "(department names come from the reference). Render a horizontal bar chart: department on the y-axis, "
    "monthly USD on the x-axis, bars sorted by cost descending, each bar labelled with its dollar value, "
    "title 'M365 monthly license cost by department'. "
    "Save the chart into the SAME directory as the uploaded CSV. "
    "Prefer matplotlib with a non-interactive backend (Agg); if it is not importable, pip install it into your "
    "session shell; if you have no network to install packages, hand-write a self-contained SVG bar chart instead. "
    "Name the file `license_cost_by_department.png` (matplotlib) or `license_cost_by_department.svg` (SVG fallback). "
    "After saving, reply with EXACTLY one line and nothing else: `CHART_FILE: <filename>`.",
    session_id=SESSION_ID,
)

m = re.search(r"CHART_FILE:\s*`?([^\s`]+)", chart_answer)
if not m:
    raise RuntimeError(f"Agent did not report a CHART_FILE line. Full reply:\n{chart_answer}")
chart_remote = m.group(1)
chart_local = f"data/{chart_remote}"

try:
    n = download_session_file(SESSION_ID, chart_remote, chart_local)
    print(f"\nDownloaded {chart_remote} -> {chart_local} ({n} bytes)")
    display(SVG(filename=chart_local) if chart_remote.endswith(".svg") else Image(filename=chart_local))
except Exception as exc:
    # Some platform builds do not serve agent-written files through the file API.
    # Fall back to having the agent return the chart inline as SVG (always works over SSE).
    print(f"\n[download] could not GET {chart_remote} ({exc}); asking the agent to inline the SVG instead...")
    svg_answer, _ = invoke(
        "Re-render that same department-cost bar chart as one self-contained SVG and output ONLY the raw "
        "SVG markup, starting with <svg and ending with </svg> - no code fences, no commentary.",
        session_id=SESSION_ID,
    )
    sm = re.search(r"<svg.*?</svg>", svg_answer, re.S)
    if not sm:
        raise RuntimeError(f"No inline SVG in fallback reply:\n{svg_answer[:600]}")
    chart_local = "data/license_cost_by_department.svg"
    with open(chart_local, "w", encoding="utf-8") as f:
        f.write(sm.group(0))
    print(f"Saved inline SVG -> {chart_local}")
    display(SVG(data=sm.group(0)))

### Optional - delete the uploaded file from the session sandbox

The file persists for the life of the session (up to 30 days or until the session is explicitly deleted). To remove just the CSV without tearing down the session:


In [ ]:
# Uncomment to delete the CSV from the session sandbox.
# delete_url = (
#     f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}/endpoint/sessions/{SESSION_ID}"
#     f"/files?api-version={API_VERSION}&path={CSV_REMOTE}"
# )
# r = requests.delete(delete_url, headers={
#     "Authorization": f"Bearer {_bearer()}",
#     "Foundry-Features": "HostedAgents=V1Preview",
# })
# r.raise_for_status()
# print(f"Deleted {CSV_REMOTE} from session {SESSION_ID}")


## Step 8 - Inspect tracing in Foundry portal

`tracing.py` maps each Copilot `SessionEvent` to an OpenTelemetry span and exports it to Application Insights. In **Foundry portal -> your project -> Tracing** you should now see one tree per invocation:

```
invoke_agent github-copilot           (SERVER, parent)
+-- execute_tool <name>                           (one per tool / skill call)
+-- chat gpt-5.4-mini                             (Tokens In/Out + Cost)
```

The `chat <model>` span is what populates the **Tokens (In)**, **Tokens (Out)**, and **Estimated Cost** columns. The CSV analytics turns will have multiple `execute_tool` spans per invocation (one per shell / awk / python call the Copilot SDK made) - those are how the agent actually answered each question.


## Step 9 - Customize the agent

Two clean extension surfaces, no code edits required:

| Knob | When to use | How |
|---|---|---|
| `src/github-copilot-invocations/system_prompt.md` | Persona / global policy that applies on every turn | Edit the file, re-run Step 3 (rebuild) + Step 4 (re-register) |
| `src/github-copilot-invocations/skills/<name>/SKILL.md` | Task-specific procedure the model discovers on demand | `mkdir skills/<name>` + write a `SKILL.md`, rebuild + re-register |

`system_prompt.md` is **appended** to the Copilot CLI's built-in system message (CLI guardrails are preserved); leaving it empty falls back to the CLI default. The bundled `skills/m365-license-analytics/SKILL.md` shows the right split: the skill carries the durable analysis *method* and the reclaim-rule definition, while volatile data (SKU costs, department names) is uploaded separately as `m365-reference.json` - so prices never live in the skill. A natural next skill would draft the deprovisioning change-request from the reclaim list.

## Cleanup

Tear down the whole resource group when you're done. The cell below is commented intentionally - uncomment when you actually want to tear down.


In [ ]:
# Uncomment to delete every resource we provisioned.
# sh(f'az group delete --name "{RESOURCE_GROUP}" --yes --no-wait')
# # The AI Services account uses soft-delete; if you want to fully purge it so the name is reusable:
# # sh(f'az cognitiveservices account purge --location "{LOCATION}" --resource-group "{RESOURCE_GROUP}" --name "{ACCOUNT_NAME}"')
